# Notebook 00: Load DIV30/DIV90 Sources Into AnnData

This notebook is now a thin driver around reusable project code. It chooses one data source, logs source availability, loads only available samples, runs QC/filtering/preprocessing from objects generated in this notebook, and saves plots to a dedicated run directory while also showing them inline.

Supported sources:

- `cellranger_filtered`
- `cellranger_raw`
- `cellbender_denoised`

Keep the default source as `cellranger_filtered` until the CellBender output set has been fully verified.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_tuple(name: str, default: tuple[str, ...]) -> tuple[str, ...]:
    raw = os.environ.get(name)
    if raw is None or not raw.strip():
        return default
    return tuple(part.strip() for part in raw.split(",") if part.strip())


# Make local package imports work whether the notebook is opened from repo root or notebooks/.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "python_notebooks" / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break

from mge_organoid_python.data_sources import (
    Notebook00SourceConfig,
    find_repo_root,
    load_dataset_result,
    resolve_data_root,
)
from mge_organoid_python.notebook00_plots import (
    PlotConfig,
    plot_embedding,
    plot_marker_panel,
    plot_qc_scatter,
    plot_qc_violin,
    plot_sample_counts,
    plot_source_availability,
)
from mge_organoid_python.notebook00_workflow import (
    PreprocessSettings,
    annotate_mad_qc,
    calculate_qc_metrics,
    concat_samples,
    filter_qc_pass_samples,
    preprocess_basic,
    qc_annotation_summary,
    run_neighbors_umap,
)

## Configuration

Change `ACTIVE_SOURCE` here. Everything below uses the objects produced by this run.

In [ ]:
REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT = resolve_data_root()

# Edit these values in the notebook, or override them in batch with environment variables.
ACTIVE_SOURCE = os.environ.get("NOTEBOOK00_ACTIVE_SOURCE", "cellranger_filtered")
TARGET_DIVS = env_tuple("NOTEBOOK00_TARGET_DIVS", ("DIV30",))
TARGET_RUN_SAMPLE_IDS = env_tuple(
    "NOTEBOOK00_TARGET_RUN_SAMPLE_IDS",
    (
        "9853-MW-1",
        "9853-MW-2",
        "9853-MW-3",
        "9853-MW-4",
        "9853-MW-5",
        "9853-MW-6",
    ),
)

# False means missing sources are logged and skipped. True means fail before loading.
STRICT_MISSING_SOURCES = env_bool("NOTEBOOK00_STRICT_MISSING_SOURCES", False)

# Set to False for source/path/report-only runs that should not load large matrices.
LOAD_MATRICES = env_bool("NOTEBOOK00_LOAD_MATRICES", True)

RUN_LABEL = os.environ.get("NOTEBOOK00_RUN_LABEL") or f"{ACTIVE_SOURCE}_{'_'.join(TARGET_DIVS).lower()}_core_samples"
RUN_DIR = DATA_ROOT / "results" / "notebook00" / RUN_LABEL
TABLE_DIR = RUN_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plot_config = PlotConfig.from_root(
    DATA_ROOT,
    run_label=RUN_LABEL,
    show=env_bool("NOTEBOOK00_SHOW_PLOTS", True),
    save=env_bool("NOTEBOOK00_SAVE_PLOTS", True),
)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("ACTIVE_SOURCE:", ACTIVE_SOURCE)
print("TARGET_DIVS:", TARGET_DIVS)
print("TARGET_RUN_SAMPLE_IDS:", TARGET_RUN_SAMPLE_IDS)
print("STRICT_MISSING_SOURCES:", STRICT_MISSING_SOURCES)
print("LOAD_MATRICES:", LOAD_MATRICES)
print("RUN_DIR:", RUN_DIR)
print("PLOT_DIR:", plot_config.output_dir)
print("TABLE_DIR:", TABLE_DIR)

## Source Availability

This cell does not load expression matrices. It reports requested samples, available paths, skipped samples, and why any sample is skipped.

In [ ]:
source_config = Notebook00SourceConfig.from_defaults(
    data_source=ACTIVE_SOURCE,
    repo_root=REPO_ROOT,
    data_root=DATA_ROOT,
    target_divs=TARGET_DIVS,
    target_run_sample_ids=TARGET_RUN_SAMPLE_IDS,
    strict_missing_matrix_dirs=STRICT_MISSING_SOURCES,
)

source_preview = load_dataset_result(source_config, load_matrices=False)
source_table = source_preview.source_table
source_summary_df = source_preview.availability_summary()

source_table.to_csv(TABLE_DIR / "source_table.tsv", sep="	", index=False)
source_summary_df.to_csv(TABLE_DIR / "source_summary.tsv", sep="	", index=False)

display(source_table)
display(source_summary_df)

plot_source_availability(source_table, plot_config)
plot_sample_counts(source_table, plot_config)

## Load Available Samples

Only samples marked `available` in `source_table` are loaded when strict mode is disabled.

In [ ]:
if LOAD_MATRICES:
    dataset_result = load_dataset_result(source_config, load_matrices=True)
    adata_names = dataset_result.adata_names
    adata_list = dataset_result.adata_list
    source_table = dataset_result.source_table
else:
    dataset_result = source_preview
    adata_names = []
    adata_list = []
    source_table = source_preview.source_table.copy()
    print("LOAD_MATRICES is False; generated source reports only and skipped AnnData loading.")

source_table.to_csv(TABLE_DIR / "loaded_source_table.tsv", sep="	", index=False)

print("Loaded samples:", dataset_result.loaded_samples)
print("Skipped samples:", dataset_result.skipped_samples)
print("Number of AnnData objects:", len(adata_list))

loaded_shape_df = pd.DataFrame(
    {
        "run_sample_id": run_sample_id,
        "n_obs": one_sample_adata.n_obs,
        "n_vars": one_sample_adata.n_vars,
        "obs_names_unique": one_sample_adata.obs_names.is_unique,
        "data_source": one_sample_adata.obs["data_source"].iloc[0] if "data_source" in one_sample_adata.obs else ACTIVE_SOURCE,
    }
    for run_sample_id, one_sample_adata in zip(adata_names, adata_list)
)
loaded_shape_df.to_csv(TABLE_DIR / "loaded_sample_shapes.tsv", sep="	", index=False)
display(loaded_shape_df)

## QC Annotation

QC annotation adds metrics and flags. It does not remove cells.

In [ ]:
if adata_list:
    calculate_qc_metrics(adata_list)
    qc_thresholds_df = annotate_mad_qc(adata_names, adata_list)
    qc_summary_df = qc_annotation_summary(adata_names, adata_list)
else:
    qc_thresholds_df = pd.DataFrame()
    qc_summary_df = pd.DataFrame()
    print("No AnnData objects loaded; skipping QC annotation.")

qc_thresholds_df.to_csv(TABLE_DIR / "qc_mad_thresholds.tsv", sep="	", index=False)
qc_summary_df.to_csv(TABLE_DIR / "qc_annotation_summary.tsv", sep="	", index=False)

display(qc_summary_df)
display(qc_thresholds_df)

## QC Filtering And Concatenation

The raw per-sample `adata_list` remains unchanged. Filtering creates retained-cell copies for analysis.

In [ ]:
APPLY_QC_FILTER = env_bool("NOTEBOOK00_APPLY_QC_FILTER", True)

if adata_list:
    if APPLY_QC_FILTER:
        analysis_names, analysis_list, qc_filter_summary_df = filter_qc_pass_samples(adata_names, adata_list)
    else:
        analysis_names, analysis_list = list(adata_names), list(adata_list)
        qc_filter_summary_df = pd.DataFrame(
            {
                "run_sample_id": analysis_names,
                "starting_n_cells": [adata.n_obs for adata in analysis_list],
                "retained_n_cells": [adata.n_obs for adata in analysis_list],
                "removed_n_cells": [0 for _ in analysis_list],
                "retained_pct": [100.0 for _ in analysis_list],
            }
        )

    combined_adata = concat_samples(analysis_names, analysis_list)
    print("combined_adata:", combined_adata.shape)
    print("combined samples:", combined_adata.obs["run_sample_id"].nunique())
else:
    analysis_names, analysis_list = [], []
    qc_filter_summary_df = pd.DataFrame()
    combined_adata = None
    print("No AnnData objects loaded; skipping QC filtering and concatenation.")

qc_filter_summary_df.to_csv(TABLE_DIR / "qc_filter_summary.tsv", sep="	", index=False)
display(qc_filter_summary_df)

## Preprocess And Embed

Run this on a compute node. The object used here is `combined_adata`, generated above.

In [ ]:
RUN_PREPROCESS = env_bool("NOTEBOOK00_RUN_PREPROCESS", True)
RUN_UMAP = env_bool("NOTEBOOK00_RUN_UMAP", True)

if combined_adata is None:
    print("No combined AnnData object; skipping preprocessing and UMAP.")
else:
    if RUN_PREPROCESS:
        preprocess_report = preprocess_basic(combined_adata, PreprocessSettings())
        pd.DataFrame([preprocess_report]).to_csv(TABLE_DIR / "preprocess_report.tsv", sep="	", index=False)
        display(pd.DataFrame([preprocess_report]))

    if RUN_UMAP:
        embedding_report = run_neighbors_umap(combined_adata)
        pd.DataFrame([embedding_report]).to_csv(TABLE_DIR / "embedding_report.tsv", sep="	", index=False)
        display(pd.DataFrame([embedding_report]))

## Plots

All plots save under `PLOT_DIR` and also display inline.

In [ ]:
if combined_adata is None:
    print("No combined AnnData object; source availability plots were already generated above.")
else:
    plot_qc_violin(combined_adata, plot_config)
    plot_qc_scatter(combined_adata, plot_config)

    if "X_umap" in combined_adata.obsm:
        plot_embedding(
            combined_adata,
            plot_config,
            basis="umap",
            color=["run_sample_id", "cell_line", "total_counts", "pct_counts_mt"],
            name="umap_sample_cellline_qc",
        )
    else:
        print("No UMAP found. Run the preprocess/embed cell first.")

## Marker Panel

Change `MARKERS` as needed. Missing markers are skipped.

In [ ]:
MARKERS = ["SOX2", "VIM", "MKI67", "EOMES", "DCX", "GAD1", "GAD2", "DLX1", "DLX2"]

if combined_adata is None:
    print("No combined AnnData object; skipping marker panel.")
else:
    markers_present = [gene for gene in MARKERS if gene in combined_adata.var_names]
    markers_missing = [gene for gene in MARKERS if gene not in combined_adata.var_names]

    pd.DataFrame({"marker": MARKERS, "present": [gene in markers_present for gene in MARKERS]}).to_csv(
        TABLE_DIR / "marker_presence.tsv",
        sep="	",
        index=False,
    )

    print("Markers present:", markers_present)
    print("Markers missing:", markers_missing)

    if "X_umap" in combined_adata.obsm:
        plot_marker_panel(combined_adata, plot_config, markers=MARKERS, name="umap_marker_panel")
    else:
        print("No UMAP found. Run the preprocess/embed cell before marker plotting.")

## Generated Objects

Main objects created by this notebook run:

- `source_table`: requested source paths and availability
- `adata_names`, `adata_list`: loaded per-sample objects
- `analysis_names`, `analysis_list`: retained-cell per-sample objects
- `combined_adata`: concatenated object for preprocessing and plots
- `TABLE_DIR`: saved TSV reports
- `plot_config.output_dir`: saved plot PNGs